03_sentiment_analysis_gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/03_sentiment_analysis_gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/03_sentiment_analysis_gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Sentiment Analysis with Gemini: Does Model Size Matter?

## What you will learn
- Use an LLM for sentiment analysis on real review data.
- Run the **same task** with a smaller model and a larger model.
- Compare their accuracy and disagreements to understand what "more intelligent" buys you.

**NLP Task**: Sentiment & Opinion Analysis — gauging attitudes and opinions in text.

**Dataset**: `rotten_tomatoes` from HuggingFace — movie reviews labelled positive/negative.
We reframe it as a product review analysis scenario.

**Key question**: When is a cheaper, faster model good enough — and when do you need a bigger one?

Expected runtime: 10–15 minutes (two model runs)
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai datasets pandas openpyxl


In [ ]:
import os
import json
import time
import pandas as pd

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Load Dataset


In [ ]:
from datasets import load_dataset

ds = load_dataset('rotten_tomatoes', split='test')
LABEL_MAP = {0: 'negative', 1: 'positive'}

import random
random.seed(42)

# Get 10 positive, 10 negative
pos_idx = [i for i in range(len(ds)) if ds[i]['label'] == 1]
neg_idx = [i for i in range(len(ds)) if ds[i]['label'] == 0]
sample_indices = random.sample(pos_idx, 10) + random.sample(neg_idx, 10)
random.shuffle(sample_indices)

sample_df = pd.DataFrame([ds[i] for i in sample_indices])
sample_df['ground_truth'] = sample_df['label'].map(LABEL_MAP)

print(f'Sample: {len(sample_df)} reviews')
print(f'Distribution: {dict(sample_df["ground_truth"].value_counts())}')
sample_df[['text', 'ground_truth']].head(5)


## The Classification Function

We use the **same prompt** for both models — this ensures any performance
difference comes from the model itself, not from the instructions.


In [ ]:
SENTIMENT_PROMPT = """Classify the sentiment of this product review as either "positive" or "negative".
Respond with ONLY the word "positive" or "negative".

Review: "{text}"

Sentiment:"""

def classify_sentiment(text: str, model_name: str) -> str:
    """Classify sentiment using the specified Gemini model."""
    prompt = SENTIMENT_PROMPT.format(text=text)
    try:
        resp = client.models.generate_content(model=model_name, contents=prompt)
        result = resp.text.strip().lower()
        if 'positive' in result:
            return 'positive'
        elif 'negative' in result:
            return 'negative'
        return result  # unexpected output — keep raw for inspection
    except Exception as e:
        return f'error: {e}'


---
## Phase 1: Smaller Model (Flash Lite)

First, we run sentiment analysis with **Gemini Flash Lite** — a smaller,
faster, cheaper model. This is the kind of model you'd use in production
when processing thousands of reviews per day.


In [ ]:
MODEL_SMALL = 'gemini-3.1-flash-lite-preview'
print(f'Running Phase 1 with: {MODEL_SMALL}')
print(f'Processing {len(sample_df)} reviews...\n')

small_results = []
for _, row in sample_df.iterrows():
    pred = classify_sentiment(row['text'], MODEL_SMALL)
    small_results.append(pred)
    time.sleep(0.3)

sample_df['pred_small'] = small_results
sample_df['correct_small'] = sample_df['pred_small'] == sample_df['ground_truth']

accuracy_small = sample_df['correct_small'].mean()
print(f'Flash Lite accuracy: {accuracy_small:.0%}')
print(f'Errors: {(~sample_df["correct_small"]).sum()} out of {len(sample_df)}')


In [ ]:
# Show the errors from the small model
errors_small = sample_df[~sample_df['correct_small']]
if len(errors_small) > 0:
    print('=== Flash Lite Errors ===')
    for _, row in errors_small.iterrows():
        print(f'  "{row["text"][:80]}..."')
        print(f'    Truth: {row["ground_truth"]}  |  Predicted: {row["pred_small"]}')
        print()
else:
    print('No errors from Flash Lite! Try running again — LLM outputs vary between runs.')


---
## Phase 2: Larger Model (Flash)

Now we run the **exact same prompt** on every review, but using
**Gemini Flash** — a more capable (and more expensive) model.

The question: does the bigger model fix the errors the smaller one made?


In [ ]:
MODEL_LARGE = 'gemini-3-flash-preview'
print(f'Running Phase 2 with: {MODEL_LARGE}')
print(f'Processing {len(sample_df)} reviews...\n')

large_results = []
for _, row in sample_df.iterrows():
    pred = classify_sentiment(row['text'], MODEL_LARGE)
    large_results.append(pred)
    time.sleep(0.3)

sample_df['pred_large'] = large_results
sample_df['correct_large'] = sample_df['pred_large'] == sample_df['ground_truth']

accuracy_large = sample_df['correct_large'].mean()
print(f'Flash accuracy: {accuracy_large:.0%}')
print(f'Errors: {(~sample_df["correct_large"]).sum()} out of {len(sample_df)}')


---
## Compare the Two Models


In [ ]:
# Where do the models agree and disagree?
sample_df['models_agree'] = sample_df['pred_small'] == sample_df['pred_large']

print('=== Model Comparison ===')
print(f'Flash Lite accuracy:  {accuracy_small:.0%}')
print(f'Flash accuracy:       {accuracy_large:.0%}')
print(f'Models agree:         {sample_df["models_agree"].mean():.0%}')
print()

# Cases where they disagree
disagreements = sample_df[~sample_df['models_agree']]
print(f'Disagreements: {len(disagreements)} reviews\n')

for _, row in disagreements.iterrows():
    print(f'  "{row["text"][:80]}..."')
    print(f'    Truth: {row["ground_truth"]}  |  Lite: {row["pred_small"]}  |  Flash: {row["pred_large"]}')
    small_right = '✓' if row['correct_small'] else '✗'
    large_right = '✓' if row['correct_large'] else '✗'
    print(f'    Lite {small_right}  |  Flash {large_right}')
    print()


In [ ]:
# Full side-by-side table
print(f'{"Review (first 55 chars)":<60} {"Truth":<10} {"Lite":<10} {"Flash":<10} {"Agree?"}')
print('-' * 100)
for _, row in sample_df.iterrows():
    text_short = row['text'][:55] + '...'
    agree = '✓' if row['models_agree'] else '✗'
    print(f'{text_short:<60} {row["ground_truth"]:<10} {row["pred_small"]:<10} {row["pred_large"]:<10} {agree}')


## Discussion Questions

1. **Did the bigger model fix the smaller model's errors?**
   Or did it introduce new ones? A bigger model isn't always better on every example.

2. **Look at the disagreements.** Read the actual review text.
   Is the ground truth even clear to you as a human reader? Some reviews are genuinely ambiguous.

3. **Cost vs. accuracy trade-off.**
   If Flash Lite gets 85% right and Flash gets 90% right, is that 5% worth
   the extra cost and latency? It depends on what you're using it for.

4. **When would you pick each model?**
   Think about: volume of reviews, cost constraints, acceptable error rate,
   and what happens when the model gets it wrong.

## Export for Your Annotation

Download this Excel file. For each review, you can annotate:
- **your_label**: What would YOU classify it as? (Sometimes ground truth is debatable.)
- **student_notes**: Any observations about why the models agreed or disagreed.


In [ ]:
export_df = sample_df[['text', 'ground_truth', 'pred_small', 'correct_small',
                        'pred_large', 'correct_large', 'models_agree']].copy()
export_df = export_df.rename(columns={
    'pred_small': 'flash_lite_prediction',
    'correct_small': 'flash_lite_correct',
    'pred_large': 'flash_prediction',
    'correct_large': 'flash_correct',
})
export_df['your_label'] = ''
export_df['student_notes'] = ''

export_path = 'sentiment_model_comparison.xlsx'
export_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')
